## API 키 설정 

In [1]:
# Load environment variables in a file called .env
# API 키 설정 
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

# Check the key
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

API key found and looks good so far!
Google API Key exists and begins AI


# 출력형식(Output Parser)

## CommaSepartedListOutputParser

In [14]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# ChatOpenAI 모델을 초기화합니다.
# temperature=0: 매번 동일한 결과를 출력하도록 창의성을 최소화합니다.
# max_tokens=2048: 모델이 생성할 수 있는 최대 토큰(단어) 수를 2048로 제한합니다.
# model_name: 사용할 OpenAI 모델을 지정합니다.
llm = ChatOpenAI(
    temperature=0,
    max_tokens=2048,
    model_name='gpt-4.1-nano',
)

# CommaSeparatedListOutputParser를 초기화합니다.
# 이 파서는 LLM이 반환한 텍스트를 쉼표(,) 기준으로 분리하여
# 파이썬 리스트(list) 형태로 변환해 줍니다.
# 예: "전북 현대, FC 서울, 울산 현대" → ["전북 현대", "FC 서울", "울산 현대"]
output_parser = CommaSeparatedListOutputParser()

# 파서가 요구하는 출력 형식 지침을 가져옵니다.
# 이 지침은 프롬프트에 포함되어 LLM이 쉼표로 구분된 형식으로
# 응답하도록 유도하는 역할을 합니다.
# 예: "Your response should be a list of comma separated values,
#      eg: `foo, bar, baz`"
format_instructions = output_parser.get_format_instructions()

# PromptTemplate을 설정합니다.
# template: LLM에 전달할 프롬프트의 뼈대입니다.
#   - {subject}: 사용자가 입력할 주제 변수
#   - {format_instructions}: 파서가 제공한 출력 형식 지침
# input_variables: 외부에서 직접 입력받을 변수 목록입니다.
# partial_variables: 미리 고정된 변수로, format_instructions처럼
#                    변경되지 않는 값을 사전에 지정합니다.
prompt = PromptTemplate(
    template="{subject}을(를) 나열해줘.\n{format_instructions}",
    input_variables=["subject"],
    partial_variables={"format_instructions": format_instructions}
)

# LLM에 전달할 주제(질문)를 정의합니다.
query = "한국의 축구팀"

# prompt.format()으로 template의 변수를 채워 최종 프롬프트 문자열을 생성합니다.
# 결과 예시:
# "한국의 축구팀을(를) 나열해줘.
#  Your response should be a list of comma separated values, ..."
formatted_prompt = prompt.format(subject=query)

# llm.invoke()로 LLM을 호출하여 응답을 받습니다.
# 반환값은 AIMessage 객체이며, 실제 텍스트는 .content 속성에 담겨 있습니다.
# 예: AIMessage(content="전북 현대, FC 서울, 울산 현대, ...")
output = llm.invoke(formatted_prompt)

# output.content로 AIMessage에서 텍스트만 추출한 뒤,
# output_parser.parse()로 쉼표 기준 분리하여 파이썬 리스트로 변환합니다.
# 예: "전북 현대, FC 서울" → ["전북 현대", "FC 서울"]
result = output_parser.parse(output.content)

# 최종 파싱된 리스트를 출력합니다.
print(result)


['전북 현대 모터스', '울산 현대', 'FC 서울', '수원 삼성 블루윙즈', '포항 스틸러스', '인천 유나이티드', '강원 FC', '대구 FC', '광주 FC', '성남 FC', '제주 유나이티드', '부산 아이파크']


## JsonOutputParser

In [15]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# ChatOpenAI 모델을 초기화합니다.
# temperature=0: 매번 동일한 결과를 출력하도록 창의성을 최소화합니다.
# max_tokens=2048: 모델이 생성할 수 있는 최대 토큰(단어) 수를 2048로 제한합니다.
# model_name: 사용할 OpenAI 모델을 지정합니다.
llm = ChatOpenAI(
    temperature=0,
    max_tokens=2048,
    model_name='gpt-4.1-nano',
)

# JsonOutputParser를 초기화합니다.
# 이 파서는 LLM이 반환한 텍스트를 JSON 형식으로 파싱하여
# 파이썬 딕셔너리(dict) 또는 리스트(list)로 변환해 줍니다.
# 예: '{"players": ["손흥민", "이강인", "김민재"]}' → {"players": ["손흥민", "이강인", "김민재"]}
output_parser = JsonOutputParser()

# 파서가 요구하는 출력 형식 지침을 가져옵니다.
# 이 지침은 프롬프트에 포함되어 LLM이 반드시 JSON 형식으로
# 응답하도록 유도하는 역할을 합니다.
# 예: "Return a JSON object."
format_instructions = output_parser.get_format_instructions()

# PromptTemplate을 설정합니다.
# template: LLM에 전달할 프롬프트의 뼈대입니다.
#   - {subject}: 사용자가 입력할 주제 변수
#   - {format_instructions}: 파서가 제공한 JSON 출력 형식 지침
# input_variables: 외부에서 직접 입력받을 변수 목록입니다.
# partial_variables: 미리 고정된 변수로, format_instructions처럼
#                    변경되지 않는 값을 사전에 지정합니다.
prompt = PromptTemplate(
    template="3명의 선수를 말해줘! {subject}.\n{format_instructions}",
    input_variables=["subject"],
    partial_variables={"format_instructions": format_instructions}
)

# LLM에 전달할 주제(질문)를 정의합니다.
query = "한국의 축구선수는?"

# prompt.format()으로 template의 변수를 채워 최종 프롬프트 문자열을 생성합니다.
# 결과 예시:
# "3명의 선수를 말해줘! 한국의 축구선수는?.
#  Return a JSON object."
formatted_prompt = prompt.format(subject=query)

# llm.invoke()로 LLM을 호출하여 응답을 받습니다.
# 반환값은 AIMessage 객체이며, 실제 텍스트는 .content 속성에 담겨 있습니다.
# 예: AIMessage(content='{"players": ["손흥민", "이강인", "김민재"]}')
output = llm.invoke(formatted_prompt)

# output.content로 AIMessage에서 텍스트만 추출한 뒤,
# output_parser.parse()로 JSON 문자열을 파이썬 딕셔너리로 변환합니다.
# 예: '{"players": ["손흥민", "이강인", "김민재"]}' → {"players": ["손흥민", "이강인", "김민재"]}
parsed_result = output_parser.parse(output.content)

# 최종 파싱된 딕셔너리(또는 리스트)를 출력합니다.
print(parsed_result)


{'players': ['손흥민', '이강인', '황희찬']}


In [30]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

# Pydantic으로 응답 스키마 정의
class AnswerSchema(BaseModel):
    answer: str = Field(description="사용자의 질문에 대한 답변")
    source: str = Field(description="출처, 웹사이트 주소")

model = ChatOpenAI(
    temperature=0,
    model_name='gpt-4.1-nano',
)

# ✅ langchain_core의 JsonOutputParser 사용
output_parser = JsonOutputParser(pydantic_object=AnswerSchema)

# 파서에서 출력 형식 지침을 가져옵니다.
format_instructions = output_parser.get_format_instructions()

# 프롬프트 템플릿 정의
prompt = PromptTemplate(
    template="answer the users question as best as possible.\n{format_instructions}\n{question}",
    input_variables=["question"],
    partial_variables={"format_instructions": format_instructions},
)

question = "대한민국의 수도는 어디인가요?"
formatted_prompt = prompt.format(question=question)

# LLM 호출
response = model.invoke(formatted_prompt)

# 파싱
parsed_result = output_parser.parse(response.content)

print(parsed_result)
print(f"답변: {parsed_result['answer']}")
print(f"출처: {parsed_result['source']}")

{'answer': '대한민국의 수도는 서울입니다.', 'source': 'https://ko.wikipedia.org/wiki/서울'}
답변: 대한민국의 수도는 서울입니다.
출처: https://ko.wikipedia.org/wiki/서울


## PydanticToolsParser

In [29]:
from langchain_core.output_parsers.openai_tools import PydanticToolsParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

# Pydantic으로 응답 스키마 정의
class AnswerSchema(BaseModel):
    answer: str = Field(description="사용자의 질문에 대한 답변")
    source: str = Field(description="출처, 웹사이트 주소")

model = ChatOpenAI(
    temperature=0,
    model_name='gpt-4.1-nano',
)

# ✅ PydanticToolsParser 초기화
# tools: 파싱할 Pydantic 스키마 클래스 리스트
output_parser = PydanticToolsParser(tools=[AnswerSchema])

# ✅ 모델에 Tool(스키마)을 바인딩
# bind_tools()로 LLM이 Function Calling을 사용하도록 설정
model_with_tools = model.bind_tools([AnswerSchema])

# 프롬프트 템플릿 정의
prompt = PromptTemplate(
    template="answer the users question as best as possible.\n{question}",
    input_variables=["question"],
)

question = "대한민국의 수도는 어디인가요?"
formatted_prompt = prompt.format(question=question)

# LLM 호출 → AIMessage (tool_calls 포함)
response = model_with_tools.invoke(formatted_prompt)

# PydanticToolsParser로 파싱 → Pydantic 객체 리스트 반환
parsed_result = output_parser.invoke(response)

print(parsed_result)
print(f"답변: {parsed_result[0].answer}")
print(f"출처: {parsed_result[0].source}")

[AnswerSchema(answer='대한민국의 수도는 서울입니다.', source='https://ko.wikipedia.org/wiki/%EB%8C%80%ED%95%9C%EB%AF%BC%EA%B5%AD')]
답변: 대한민국의 수도는 서울입니다.
출처: https://ko.wikipedia.org/wiki/%EB%8C%80%ED%95%9C%EB%AF%BC%EA%B5%AD
